# 05 · `fact_inventario` — foto diaria de stock
**IDY1102 · Experiencia 3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Victor Navarrete] |
| Tabla destino | `fact_inventario` |
| Grano | una fila por **día × producto × sucursal** (foto al cierre del día) |
| Medidas | `stock_unidades`, `valor_stock`, `unidades_merma` (semiaditivas) · `quiebre` (0/1) |
| Depende de | `dim_fecha`, `dim_producto` (SCD2 + costo), `dim_sucursal` — notebooks 02 y 04 |
| Fuente | `inventario_wms.csv` (WMS) |
| Alimenta a | K4 (quiebre de stock) y K6 (rotación de inventario) de la ficha de KPI |

> **Semiaditiva:** el stock se puede sumar entre productos o sucursales, pero **no en el tiempo** (sumar 30 fotos
> diarias no da el stock del mes). Para un periodo se usa el promedio o la última foto.

## 0. Setup

In [ ]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "fact_inventario"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

## 1. Extract
Lee el archivo del WMS como texto y lo guarda **sin cambios** en la capa Bronze.

In [ ]:
def extraer_inventario() -> pd.DataFrame:
    df = extract.leer_csv(cfg.data_dir / "inventario_wms.csv")
    medallion.guardar_capa(df, "bronze", "inventario")
    return df

## 2. Transform
Reglas de calidad que traen los datos de ejemplo:

| Problema | Dimensión de calidad | Regla |
|---|---|---|
| Fechas en otro formato (`dd/mm/aaaa`) | Validez | `q.parsear_fechas`; lo imposible se rechaza |
| SKU en minúsculas | Consistencia | `q.normalizar_sku` |
| Stock vacío o negativo | Completitud / Validez | se rechazan (`stock_vacio`, `stock_negativo`) |
| Foto repetida (batch duplicado) | Unicidad | una por `(fecha, sucursal, SKU)` |
| Sucursal `S99` que no existe | Integridad referencial | se rechaza (`sucursal_desconocida`) |
| Días sin entrega del WMS | Completitud | no se inventan datos: se mide en Validación |

In [ ]:
def transformar_inventario(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    rechazos = []

    df["fecha"] = q.parsear_fechas(df["fecha"])
    df, r = q.separar_rechazos(df, df["fecha"].isna(), "fecha_invalida"); rechazos.append(r)

    df["sku"] = q.normalizar_sku(df["sku"])
    df["id_sucursal"] = q.normalizar_texto(df["id_sucursal"], mayusculas=True)
    for col in ["stock_unidades", "unidades_merma"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    df, r = q.separar_rechazos(df, df["stock_unidades"].isna(), "stock_vacio"); rechazos.append(r)
    df, r = q.separar_rechazos(df, (df["stock_unidades"] < 0) | (df["unidades_merma"] < 0), "stock_negativo"); rechazos.append(r)
    df["unidades_merma"] = df["unidades_merma"].fillna(0)

    df, r = q.separar_rechazos(df, df.duplicated(["fecha", "id_sucursal", "sku"], keep="first"), "foto_duplicada"); rechazos.append(r)

    sucursales = db.mapa_claves(engine, "dim_sucursal", "id_sucursal", "sucursal_key")
    df, r = q.separar_rechazos(df, ~df["id_sucursal"].isin(sucursales), "sucursal_desconocida"); rechazos.append(r)
    skus = set(db.leer_tabla(engine, "dim_producto")["sku"])
    df, r = q.separar_rechazos(df, ~df["sku"].isin(skus), "sku_desconocido"); rechazos.append(r)
    return df, q.consolidar_rechazos(*rechazos)

## 3. Claves sustitutas y medidas
Cada foto toma la **versión del producto vigente ese día** (SCD2) y su costo unitario neto:
`valor_stock = stock_unidades × costo_unitario` y `quiebre = 1` cuando no queda stock.

In [ ]:
COLUMNAS_HECHOS = ["fecha_key", "producto_key", "sucursal_key", "stock_unidades", "valor_stock",
                   "unidades_merma", "quiebre"]

def asignar_claves(inv: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    v = inv.copy()
    v["fecha_key"] = v["fecha"].dt.strftime("%Y%m%d").astype(int)
    dim_producto = db.leer_tabla(engine, "dim_producto")
    v["producto_key"] = db.asignar_clave_scd2(v, dim_producto, "sku", "fecha", "producto_key")
    v["sucursal_key"] = v["id_sucursal"].map(db.mapa_claves(engine, "dim_sucursal", "id_sucursal", "sucursal_key")).astype(int)
    # Un producto que aún no existía en el catálogo ese día no puede tener foto de stock (y chocaría en la PK)
    v, rech = q.separar_rechazos(v, v["producto_key"] == -1, "producto_no_vigente")
    costo = v["producto_key"].map(dim_producto.set_index("producto_key")["costo_unitario"])
    v["valor_stock"] = (v["stock_unidades"] * costo).round(0)
    v["quiebre"] = (v["stock_unidades"] == 0).astype(int)
    for col in ["stock_unidades", "unidades_merma"]:
        v[col] = v[col].astype(int)
    return v, rech

## 4. Load
Carga idempotente por `(fecha_key, producto_key, sucursal_key)` y ejecución con auditoría.

In [ ]:
def cargar_hechos(hechos: pd.DataFrame, rechazos: pd.DataFrame) -> dict:
    assert set(COLUMNAS_HECHOS) <= set(hechos.columns), "asignar_claves() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, PROCESO, rechazos)
    return db.insertar_si_no_existe(engine, schema.fact_inventario, hechos[COLUMNAS_HECHOS],
                                    ["fecha_key", "producto_key", "sucursal_key"])

In [ ]:
with db.Ejecucion(engine, PROCESO) as run:
    crudo = extraer_inventario()
    run.leidas = len(crudo)
    validas, rech_calidad = transformar_inventario(crudo)
    medallion.guardar_capa(validas, "silver", "inventario")
    hechos, rech_vigencia = asignar_claves(validas)
    rechazos = q.consolidar_rechazos(rech_calidad, rech_vigencia)
    resumen = cargar_hechos(hechos, rechazos)
    run.cargadas = resumen["insertadas"]
    run.rechazadas = len(rechazos)
resumen

## 5. Validación
Conciliación de conteos, reglas del modelo y completitud de la foto diaria.

In [ ]:
def validar_hechos():
    n_leidas, n_cargables, n_rech = len(crudo), len(hechos), len(rechazos)
    assert n_leidas == n_cargables + n_rech, f"No concilia: {n_leidas} leídas ≠ {n_cargables} cargables + {n_rech} rechazadas"
    cargados = db.leer_tabla(engine, "fact_inventario")
    assert len(cargados) == n_cargables, "lo cargado no coincide con lo esperado"
    assert (cargados["stock_unidades"] >= 0).all(), "hay stock negativo en el DW"
    assert ((cargados["stock_unidades"] == 0).astype(int) == cargados["quiebre"]).all(), "quiebre inconsistente con el stock"
    assert (cargados[["producto_key", "sucursal_key"]] != -1).all().all(), "hay fotos con producto o sucursal 'No informado'"
    # Completitud: (fecha, sucursal) con menos filas que el máximo de ese día (rechazos o días que el WMS no entregó)
    por_dia = cargados.groupby(["fecha_key", "sucursal_key"]).size()
    incompletos = (por_dia < por_dia.groupby(level=0).transform("max")).sum()
    print(f"Conciliación OK · {n_leidas} = {n_cargables} + {n_rech} · pares (fecha, sucursal) incompletos: {incompletos}")

validar_hechos()

In [ ]:
# Idempotencia: volver a cargar las mismas fotos no inserta nada
filas_antes = len(db.leer_tabla(engine, "fact_inventario"))
r2 = cargar_hechos(hechos, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "fact_inventario")) == filas_antes and r2["insertadas"] == 0
print("Idempotencia OK:", r2)

## 6. Evidencia
La consulta de K4 (porcentaje de quiebre por semana y categoría) demuestra que el modelo responde la pregunta del negocio.

In [ ]:
consulta_k4 = """
SELECT d.anio_semana, p.categoria,
       ROUND(100.0 * SUM(f.quiebre) / COUNT(*), 1) AS pct_quiebre
FROM fact_inventario f
JOIN dim_fecha d    ON d.fecha_key = f.fecha_key
JOIN dim_producto p ON p.producto_key = f.producto_key
GROUP BY d.anio_semana, p.categoria
ORDER BY d.anio_semana
"""
with engine.connect() as conn:
    k4 = pd.read_sql(consulta_k4, conn)
k4.pivot(index="anio_semana", columns="categoria", values="pct_quiebre").tail(12)

In [ ]:
# Valor del stock de la ÚLTIMA foto por categoría (semiaditiva: no se suma en el tiempo)
consulta_stock = """
SELECT p.categoria, SUM(f.stock_unidades) AS unidades, SUM(f.valor_stock) AS valor_stock_clp
FROM fact_inventario f
JOIN dim_producto p ON p.producto_key = f.producto_key
WHERE f.fecha_key = (SELECT MAX(fecha_key) FROM fact_inventario)
GROUP BY p.categoria
"""
with engine.connect() as conn:
    display(pd.read_sql(consulta_stock, conn))
rech = db.leer_tabla(engine, "auditoria_rechazos")
rech[rech["proceso"] == PROCESO].groupby("motivo_rechazo").size().rename("filas").to_frame()

## 7. Decisiones de diseño (completar)
- **Grano diario por sucursal:** permite medir quiebre semanal (K4) y rotación (K6) sin perder el detalle.
- **`quiebre` como medida 0/1:** su promedio es el % de producto×sucursal×día sin stock; se acota a productos vigentes porque solo existen fotos de productos del catálogo.
- **El centro de distribución online (S00) tiene stock propio:** las ventas e-commerce salen de ahí.
- **Costo en `dim_producto` (SCD1):** el costo se sobrescribe y no justifica versionar la dimensión; `valor_stock` queda congelado al cargar.
- ¿Qué pasaría con el K4 si se contaran también productos sin demanda? (completar con el equipo)